### Reading XML data with an inferred schema

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = (SparkSession.builder
        .appName("read-json-data-01")
        .master("spark://spark-master:7077")
        .config("spark.executor.memory", "512m")
        .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/24 10:49:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = (spark.read.format("json")
     .option("multiLine", "true")
     .load("../data/nobel_prizes.json"))

In [4]:
df.printSchema()

root
 |-- category: string (nullable = true)
 |-- laureates: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- firstname: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- motivation: string (nullable = true)
 |    |    |-- share: string (nullable = true)
 |    |    |-- surname: string (nullable = true)
 |-- overallMotivation: string (nullable = true)
 |-- year: string (nullable = true)



In [5]:
df.show(2)

+---------+--------------------+-----------------+----+
| category|           laureates|overallMotivation|year|
+---------+--------------------+-----------------+----+
|chemistry|[{Carolyn, 1015, ...|             null|2022|
|economics|[{Ben, 1021, "for...|             null|2022|
+---------+--------------------+-----------------+----+
only showing top 2 rows



In [6]:
df_flattened = (
    df
    .withColumn("laureates", explode(col("laureates")))
    .select(
        col("category"),
        col("year"),
        col("overallMotivation"),
        col("laureates.id"),
        col("laureates.firstname"),
        col("laureates.surname"),
        col("laureates.share"),
        col("laureates.motivation"),
    )
)

In [7]:
df_flattened.show(2, truncate=False)

+---------+----+-----------------+----+---------+--------+-----+--------------------------------------------------------------------+
|category |year|overallMotivation|id  |firstname|surname |share|motivation                                                          |
+---------+----+-----------------+----+---------+--------+-----+--------------------------------------------------------------------+
|chemistry|2022|null             |1015|Carolyn  |Bertozzi|3    |"for the development of click chemistry and bioorthogonal chemistry"|
|chemistry|2022|null             |1016|Morten   |Meldal  |3    |"for the development of click chemistry and bioorthogonal chemistry"|
+---------+----+-----------------+----+---------+--------+-----+--------------------------------------------------------------------+
only showing top 2 rows



In [8]:
df_flattened.printSchema()

root
 |-- category: string (nullable = true)
 |-- year: string (nullable = true)
 |-- overallMotivation: string (nullable = true)
 |-- id: string (nullable = true)
 |-- firstname: string (nullable = true)
 |-- surname: string (nullable = true)
 |-- share: string (nullable = true)
 |-- motivation: string (nullable = true)



In [9]:
df_flattened.schema

StructType([StructField('category', StringType(), True), StructField('year', StringType(), True), StructField('overallMotivation', StringType(), True), StructField('id', StringType(), True), StructField('firstname', StringType(), True), StructField('surname', StringType(), True), StructField('share', StringType(), True), StructField('motivation', StringType(), True)])

In [16]:
df_flattened_with_schema = (
    spark.read.format("json")
    .schema(df_flattened.schema)
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "corrupt_record")
    .load("../data/nobel_prizes.json"))

In [17]:
df_flattened_with_schema.show(2, truncate=False)

+---------+----+-----------------+----+---------+-------+-----+----------+
|category |year|overallMotivation|id  |firstname|surname|share|motivation|
+---------+----+-----------------+----+---------+-------+-----+----------+
|chemistry|2022|null             |null|null     |null   |null |null      |
|economics|2022|null             |null|null     |null   |null |null      |
+---------+----+-----------------+----+---------+-------+-----+----------+
only showing top 2 rows



In [31]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

json_schema = StructType(
    [
     StructField('year', StringType(), True),
     StructField('category', StringType(), True),
     StructField('laureates', ArrayType(StructType(
         [StructField('firstname', StringType(), True), 
          StructField('id', StringType(), True), 
          StructField('motivation', StringType(), True), 
          StructField('share', StringType(), True), 
          StructField('surname', StringType(), True)
          ]), True), True),
     StructField('motivation', StringType(), True)
     ])

In [32]:
df_flattened_with_schema = (
    spark.read.format("json")
    .schema(json_schema)
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "corrupt_record")
    .load("../data/nobel_prizes.json"))

In [33]:
df_flattened_with_schema

DataFrame[year: string, category: string, laureates: array<struct<firstname:string,id:string,motivation:string,share:string,surname:string>>, motivation: string]

In [35]:
df_flattened_with_schema.show(1, truncate=False)

+----+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|year|category |laureates                                                                                                                                                                                                                                                                                              |motivation|
+----+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|2022|chemistry|[{Carolyn, 1

In [47]:
from pyspark.sql.functions import explode

# Define your schema (you already have this)
json_schema = StructType(
    [
     StructField('year', StringType(), True),
     StructField('category', StringType(), True),
     StructField('laureates', ArrayType(StructType(
         [StructField('firstname', StringType(), True), 
          StructField('id', StringType(), True), 
          StructField('motivation', StringType(), True), 
          StructField('share', StringType(), True), 
          StructField('surname', StringType(), True)
          ]), True), True),
     StructField('motivation', StringType(), True), 
     ])

# Read the JSON file with the schema
df_flattened_with_schema = (
    spark.read.format("json")
    .schema(json_schema)
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "corrupt_record")
    .load("../data/nobel_prizes.json")
)

# Explode the 'laureates' array to flatten the data
df_flattened = df_flattened_with_schema.withColumn("laureate", explode("laureates"))
df_flattened = df_flattened.drop("laureates")

# Select individual fields from the exploded laureates struct
df_flattened = df_flattened.withColumn("firstname", df_flattened["laureate.firstname"]) \
                           .withColumn("id", df_flattened["laureate.id"]) \
                           .withColumn("motivation", df_flattened["laureate.motivation"]) \
                           .withColumn("share", df_flattened["laureate.share"]) \
                           .withColumn("surname", df_flattened["laureate.surname"])

# Drop the original 'laureate' column
df_flattened = df_flattened.drop("laureate")

# Show the flattened DataFrame
df_flattened.show(2, truncate=False)


+----+---------+--------------------------------------------------------------------+---------+----+-----+--------+
|year|category |motivation                                                          |firstname|id  |share|surname |
+----+---------+--------------------------------------------------------------------+---------+----+-----+--------+
|2022|chemistry|"for the development of click chemistry and bioorthogonal chemistry"|Carolyn  |1015|3    |Bertozzi|
|2022|chemistry|"for the development of click chemistry and bioorthogonal chemistry"|Morten   |1016|3    |Meldal  |
+----+---------+--------------------------------------------------------------------+---------+----+-----+--------+
only showing top 2 rows



### `get_json_object()` and `json_tuple()` functions

### `flatten()` and `collect_list()` functions